# 2. Segmentation Results Explorer

**Purpose:** Load and visualize results from each segmentation method after SLURM jobs complete.  
**Container:** Must be run inside `seg_sin_V1.sif`.

Each method writes:
- `{sample_id}.h5ad` — AnnData with cell × gene counts
- Explorer-compatible zarr files
- `run_metadata_{method}.json` — timing and parameters

In [ ]:
import os
import json
import yaml
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import sopa
import spatialdata
import matplotlib.pyplot as plt
from pathlib import Path

sc.settings.set_figure_params(dpi=100, frameon=False)
%matplotlib inline

In [ ]:
# Load config
CONFIG_PATH = "../config/pipeline_config.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

SAMPLE_ID = cfg["sample"]["id"]
BASE_OUT = Path(cfg["paths"]["base_output_dir"]) / SAMPLE_ID

# Discover which methods have completed
METHODS = ["proseg", "baysor", "cellpose", "bidcell"]
available = {}
for m in METHODS:
    h5ad = BASE_OUT / m / f"{SAMPLE_ID}.h5ad"
    if h5ad.exists():
        available[m] = h5ad
        print(f"  ✓ {m:12s} → {h5ad}")
    else:
        print(f"  ✗ {m:12s}   (not found)")

print(f"\n{len(available)} method(s) ready for exploration")

## Load all available results

In [ ]:
adatas = {}
for method, path in available.items():
    adata = sc.read_h5ad(path)
    adata.obs["method"] = method
    adatas[method] = adata
    print(f"{method:12s}: {adata.n_obs:>8,} cells × {adata.n_vars:>5,} genes")

## Run metadata (timing, resources)

In [ ]:
run_info = []
for method in available:
    meta_path = BASE_OUT / method / f"run_metadata_{method}.json"
    if meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
        run_info.append({
            "method": method,
            "elapsed_min": meta.get("elapsed_minutes"),
            "job_id": meta.get("slurm_job_id"),
            "cpus": meta.get("slurm_cpus"),
            "timestamp": meta.get("timestamp"),
        })

if run_info:
    display(pd.DataFrame(run_info))

## Per-method deep dive

Select a method below to explore in detail.

In [ ]:
# ── Pick a method ──
METHOD = "proseg"  # Change to: "baysor", "cellpose", "bidcell"

assert METHOD in adatas, f"{METHOD} not available. Choose from: {list(adatas.keys())}"
adata = adatas[METHOD].copy()
print(f"\nExploring: {METHOD}")
print(f"  {adata.n_obs:,} cells × {adata.n_vars:,} genes")
adata

In [ ]:
# Basic QC metrics
sc.pp.calculate_qc_metrics(adata, inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"{METHOD} — QC Distributions", fontsize=14)

axes[0].hist(adata.obs["total_counts"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Total Counts")
axes[0].set_ylabel("Cells")
axes[0].axvline(adata.obs["total_counts"].median(), color="red", ls="--", label="median")
axes[0].legend()

axes[1].hist(adata.obs["n_genes_by_counts"], bins=50, color="coral", edgecolor="white")
axes[1].set_xlabel("Genes Detected")

axes[2].scatter(adata.obs["total_counts"], adata.obs["n_genes_by_counts"], s=1, alpha=0.2)
axes[2].set_xlabel("Total Counts")
axes[2].set_ylabel("Genes Detected")

plt.tight_layout()
plt.show()

## Quick clustering (for visualization)

In [ ]:
# Standard scanpy workflow
adata_proc = adata.copy()

# Normalize + log
sc.pp.normalize_total(adata_proc, target_sum=1e4)
sc.pp.log1p(adata_proc)

# HVGs + PCA + neighbors + UMAP + Leiden
sc.pp.highly_variable_genes(adata_proc, n_top_genes=2000)
sc.tl.pca(adata_proc, n_comps=30)
sc.pp.neighbors(adata_proc, n_pcs=30)
sc.tl.umap(adata_proc)
sc.tl.leiden(adata_proc, resolution=0.5)

sc.pl.umap(adata_proc, color="leiden", title=f"{METHOD} — Leiden Clusters")

## Spatial visualization
If spatial coordinates are available in `.obsm`, plot clusters in space.

In [ ]:
# Check for spatial coordinates
spatial_key = None
for key in ["spatial", "X_spatial"]:
    if key in adata_proc.obsm:
        spatial_key = key
        break

if spatial_key:
    fig, ax = plt.subplots(figsize=(12, 10))
    coords = adata_proc.obsm[spatial_key]
    scatter = ax.scatter(
        coords[:, 0], coords[:, 1],
        c=adata_proc.obs["leiden"].astype(int),
        cmap="tab20", s=0.5, alpha=0.6,
    )
    ax.set_aspect("equal")
    ax.set_title(f"{METHOD} — Leiden clusters in space")
    ax.invert_yaxis()
    plt.colorbar(scatter, ax=ax, label="Cluster")
    plt.tight_layout()
    plt.show()
else:
    print("No spatial coordinates found in .obsm")
    print(f"Available keys: {list(adata_proc.obsm.keys())}")

## Layers inspection
Check what count layers are available (especially for ProSeg expected counts).

In [ ]:
print("Layers:")
for key in adata.layers:
    layer = adata.layers[key]
    if hasattr(layer, 'toarray'):
        print(f"  {key}: sparse {layer.shape}, nnz={layer.nnz:,}, dtype={layer.dtype}")
    else:
        print(f"  {key}: dense {layer.shape}, dtype={layer.dtype}")

print(f"\nX: shape={adata.X.shape}, dtype={adata.X.dtype}")
if hasattr(adata.X, 'nnz'):
    sparsity = 1 - (adata.X.nnz / (adata.n_obs * adata.n_vars))
    print(f"   sparsity={sparsity:.2%}")

---
## Next: Cross-method comparison

See **notebook 03** for side-by-side QC and method benchmarking.